# Fase 2B — Machine Learning: Hard Voting LR + RF + XGBoost (IroSvA)

**Nested CV 5×3 agrupada · TF-IDF 10k · F1-Macro**

Este notebook evalúa un **Hard Voting de tres modelos**: LR + RF + XGBoost.

La combinación se fija **a priori**: gana la clase elegida por al menos 2 de los 3 modelos.
Al utilizar tres modelos no existen empates.

No se selecciona el mejor subconjunto usando los resultados Outer.
Para ahorrar cómputo, no se repite ninguna búsqueda: para cada variante y Outer Fold se reutiliza la configuración
(preprocesamiento, n-gramas e hiperparámetros) que el Inner CV del experimento 10k eligió para cada modelo, leída de
`outer_folds_ml.csv`, y cada modelo se vuelve a ajustar únicamente sobre su Outer Train. Cada modelo usa su propio preprocesamiento.
Requiere haber ejecutado antes `ML_TFIDF_10k_GroupNestedCV_5x3_CORREGIDO.ipynb` con el mismo `MODO`. Los resultados se guardan en
`../data/resultados_ml_hard_voting_tfidf10k/`.

## 1. Librerías y entorno

Importaciones, stopwords NLTK y versiones de los paquetes utilizados.

In [1]:
import ast
import json
import time
import html
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import optuna
import sklearn
import xgboost
import nltk
from nltk.corpus import stopwords
from IPython.display import display, HTML

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)
from xgboost import XGBClassifier

# Lista original del notebook LR/RF. Se usa la misma para los tres modelos.
try:
    STOP_WORDS = stopwords.words("spanish")
except LookupError:
    if not nltk.download("stopwords", quiet=True):
        raise RuntimeError(
            "Instala el corpus NLTK de stopwords para reproducir el TF-IDF original."
        )
    STOP_WORDS = stopwords.words("spanish")

optuna.logging.set_verbosity(optuna.logging.WARNING)

VERSIONES = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "optuna": optuna.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))

Salida: ../data\resultados_ml_hard_voting_tfidf10k


## 2. Configuración

Rutas, modo de ejecución y parámetros del protocolo. `MODO` debe coincidir con el usado en el notebook ML: en `"prueba"` se leen los resultados de `…/resultados_ml_groupcv_tfidf_lr_rf_10k/prueba/` y se escribe en `…/resultados_ml_hard_voting_tfidf10k/prueba/`.

El test oficial solo puede ejecutarse en modo completo, con `EJECUTAR_TEST_OFICIAL=True` y `CONFIRMACION_TEST="TEST_OFICIAL_CONGELADO"`.

In [ ]:
DATA_DIR = Path("../data")
RANDOM_STATE = 42
MODELOS = ["LR", "RF", "XGBoost"]
MODELOS_LR_RF = ["LR", "RF"]
PREPROCESAMIENTOS = {"normal": "", "stem": "_stem", "lemma": "_lemma"}
FEATURE_COLS = ["n_exc", "n_int", "n_may", "n_emo", "n_ris", "n_neg", "n_elo", "n_com", "n_pun"]

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3.
# ------------------------------------------------------------
MODO = "completo"

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# Resultados YA obtenidos por el notebook ML (mismo MODO).
BASE_OUT_DIR = DATA_DIR / "resultados_ml_groupcv_tfidf_lr_rf_10k"
VOTING_OUT_DIR = DATA_DIR / "resultados_ml_hard_voting_tfidf10k"
if MODO == "prueba":
    BASE_OUT_DIR = BASE_OUT_DIR / "prueba"
    VOTING_OUT_DIR = VOTING_OUT_DIR / "prueba"
BASE_MODELS_DIR = BASE_OUT_DIR / "modelos_finales"
VOTING_OUT_DIR.mkdir(parents=True, exist_ok=True)

N_JOBS_XGB = -1
NGRAMAS_OPCIONES = {"uni": (1, 1), "uni_bi": (1, 2)}

# Test oficial: solo con el protocolo congelado y en modo completo.
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

print(f"[CONFIG][HardVoting] modo={MODO} | variantes={VARIANTES} | modelos={MODELOS}")
print(f"[CONFIG][HardVoting] Nested={N_OUTER}×{N_INNER} | regla=mayoría 2 de 3")
print(f"[CONFIG][HardVoting] resultados ML={BASE_OUT_DIR.resolve()}")
print(f"[CONFIG][HardVoting] salida={VOTING_OUT_DIR.resolve()}")

## 3. Carga de datos, grupos y folds
Se reutiliza exactamente la lógica de agrupación del notebook 10k.

In [2]:
# ============================================================
# 3. CARGA, INTEGRIDAD Y GRUPOS SIN FUGA ENTRE FOLDS
# ============================================================
GRUPOS_TRAIN = {}  # Variante -> np.array; solo contiene información del TRAIN
AUDITORIA_GRUPOS = []
AUDITORIA_CONFLICTOS = []
AUDITORIA_PARTICIONES = []


class UnionFind:
    """Componentes conexos: unión por rango con compresión de caminos."""

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def cargar_split(variante, prep, split="train"):
    """Carga el CSV preprocesado de una variante, representación y split."""
    if variante not in VARIANTES or prep not in PREPROCESAMIENTOS:
        raise ValueError(f"Variante/preprocessing no reconocido: {variante}/{prep}")
    ruta = DATA_DIR / f"{split}_clean{PREPROCESAMIENTOS[prep]}_{variante}.csv"
    if not Path(ruta).is_file():
        raise FileNotFoundError(f"Falta el CSV: {ruta}")
    # Preservar los IDs originales: no forzarles una conversión numérica.
    return pd.read_csv(ruta, dtype={"ID": "string"}).reset_index(drop=True)


def preparar_xy(df):
    """Separa texto + 9 rasgos (X) y etiqueta (y)."""
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS
    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).reset_index(drop=True)
    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)
    return X.reset_index(drop=True), y


def texto_clave(serie):
    """Normalización usada solo para comparar textos al agrupar (no altera el texto del modelo)."""
    # Misma normalización de COMPARACIÓN que la auditoría previa; no modifica el texto del modelo.
    return serie.fillna("").astype(str).str.casefold().str.strip()


def construir_grupos(variante, dfs_por_prep, y):
    """Componentes conexos: agrupa un mismo texto tras normal, stem O lemma.
    No deduplica, no usa las etiquetas para construir grupos y conserva rasgos originales.
    Añade coincidencias de MESSAGE original exacto, si está disponible.
    """
    n = len(y)
    uf = UnionFind(n)

    for prep, df in dfs_por_prep.items():
        primero = {}
        for i, clave in enumerate(texto_clave(df["MESSAGE_CLEAN"])):
            if not clave:
                raise ValueError(
                    f"{variante}/{prep}: texto limpio vacío en fila {i}; revisar antes de agrupar"
                )
            if clave in primero:
                uf.union(primero[clave], i)
            else:
                primero[clave] = i
    # Si dos textos originales son idénticos, deben permanecer en el mismo grupo.
    df_ref = dfs_por_prep["normal"]
    if "MESSAGE" in df_ref:
        primero_raw = {}
        for i, clave in enumerate(texto_clave(df_ref["MESSAGE"])):
            if not clave:
                continue
            if clave in primero_raw:
                uf.union(primero_raw[clave], i)
            else:
                primero_raw[clave] = i

    miembros = {}
    for i in range(n):
        miembros.setdefault(uf.find(i), []).append(i)
    # Id de grupo basado en el menor índice original; determinista entre ambos notebooks.
    ids = {ra: min(idx) for ra, idx in miembros.items()}
    grupos = np.array([ids[uf.find(i)] for i in range(n)], dtype=int)
    duplicados = {ids[ra]: idx for ra, idx in miembros.items() if len(idx) > 1}
    conflictivos = [(g, idx) for g, idx in duplicados.items() if y.iloc[idx].nunique() > 1]
    if conflictivos:
        warnings.warn(
            f"{variante.upper()}: {len(conflictivos)} grupo(s) con etiquetas contradictorias. "
            "Se conservan las etiquetas originales y se mantienen juntos todos sus registros. "
            "Revisar manualmente antes de interpretar resultados.",
            stacklevel=2,
        )
    for g, idx in conflictivos:
        for i in idx:
            AUDITORIA_CONFLICTOS.append(
                {
                    "variante": variante,
                    "grupo": int(g),
                    "fila_0base": int(i),
                    "id_original": str(df_ref.iloc[i]["ID"]) if "ID" in df_ref else "",
                    "etiqueta": int(y.iloc[i]),
                    "texto_original": str(df_ref.iloc[i]["MESSAGE"]) if "MESSAGE" in df_ref else "",
                    "texto_normal": str(df_ref.iloc[i]["MESSAGE_CLEAN"]),
                }
            )
    AUDITORIA_GRUPOS.append(
        {
            "variante": variante,
            "n_train": n,
            "n_grupos": len(miembros),
            "n_grupos_multiples": len(duplicados),
            "n_filas_en_grupos_multiples": sum(map(len, duplicados.values())),
            "tamano_grupo_max": max(map(len, miembros.values())),
            "n_grupos_etiquetas_contradictorias": len(conflictivos),
            "criterio": "union_texto_normal_stem_lemma_y_original_exacto",
        }
    )
    return grupos


def cargar_todos_preps(variante, split="train"):
    """Carga normal/stem/lemma de una variante, verifica que estén alineados y construye los grupos."""
    dfs = {p: cargar_split(variante, p, split=split) for p in PREPROCESAMIENTOS}
    ref = dfs["normal"]
    X_ref, y_ref = preparar_xy(ref)
    datos = {"normal": X_ref}
    for prep, df in dfs.items():
        if prep == "normal":
            continue
        X, y = preparar_xy(df)
        if len(df) != len(ref) or not y.equals(y_ref):
            raise ValueError(f"{variante}/{prep}: diferente número de filas o etiquetas")
        for c in ["ID", "MESSAGE"]:
            if c in ref.columns and c not in df.columns:
                raise ValueError(f"{variante}/{prep}: falta columna de alineación {c}")
            if c in ref.columns and not ref[c].astype(str).equals(df[c].astype(str)):
                raise ValueError(f"{variante}/{prep}: columna {c} desalineada")
        if not X[FEATURE_COLS].equals(X_ref[FEATURE_COLS]):
            raise ValueError(f"{variante}/{prep}: rasgos lingüísticos no alineados")
        datos[prep] = X
    if split == "train" and variante not in GRUPOS_TRAIN:
        GRUPOS_TRAIN[variante] = construir_grupos(variante, dfs, y_ref)
    return datos, y_ref


def grupos_train(variante):
    """Devuelve los grupos del train de una variante (los construye si aún no existen)."""
    if variante not in GRUPOS_TRAIN:
        cargar_todos_preps(variante, split="train")
    return GRUPOS_TRAIN[variante]


def verificar_particion(grupos, a, b, variante, nivel, outer, inner=None):
    """Comprueba que train y validación no compartan grupos y registra la partición."""
    solapamiento = set(grupos[a]) & set(grupos[b])
    if solapamiento:
        raise AssertionError(
            f"FUGA: {variante}/{nivel}/{outer}/{inner} grupos compartidos: {solapamiento}"
        )
    AUDITORIA_PARTICIONES.append(
        {
            "variante": variante,
            "nivel": nivel,
            "outer_fold": outer,
            "inner_fold": inner,
            "n_train": len(a),
            "n_validacion": len(b),
            "n_grupos_compartidos": 0,
        }
    )


def crear_splits_outer(y, grupos, variante="?"):
    """Outer CV agrupada y estratificada."""
    cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "outer", fold)
    return splits


def crear_splits_inner(y, grupos, outer_fold, variante="?"):
    """Inner CV agrupada dentro de un Outer Train (semilla RANDOM_STATE + outer_fold)."""
    cv = StratifiedGroupKFold(
        n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE + outer_fold
    )
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "inner", outer_fold, fold)
    return splits


def crear_splits_final(y, grupos, variante="?"):
    """CV agrupada sobre todo el train para la búsqueda final."""
    cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "final", 0, fold)
    return splits


# ============================================================
# AUDITORÍA PREVIA — SOLO TRAIN Y FOLDS
# ============================================================
for v in VARIANTES:
    _, etiquetas = cargar_todos_preps(v, split="train")
    gv = grupos_train(v)

    for o, (a, b) in enumerate(crear_splits_outer(etiquetas, gv, v), 1):
        crear_splits_inner(etiquetas.iloc[a].reset_index(drop=True), gv[a], o, v)

    crear_splits_final(etiquetas, gv, v)


df_auditoria_grupos = pd.DataFrame(AUDITORIA_GRUPOS)

df_auditoria_particiones = pd.DataFrame(AUDITORIA_PARTICIONES).drop_duplicates()

df_auditoria_conflictos = pd.DataFrame(AUDITORIA_CONFLICTOS)

print("[AUDIT][ML] Grupos del train")
display(df_auditoria_grupos)

print("\n[AUDIT][ML] Grupos compartidos entre train y validación (todos deben ser 0):")
display(
    df_auditoria_particiones.groupby(["variante", "nivel"], as_index=False)[
        "n_grupos_compartidos"
    ].max()
)

pd.DataFrame(AUDITORIA_GRUPOS).to_csv(VOTING_OUT_DIR / "auditoria_grupos.csv", index=False)

df_auditoria_particiones.to_csv(VOTING_OUT_DIR / "auditoria_folds.csv", index=False)

df_auditoria_conflictos.to_csv(
    VOTING_OUT_DIR / "auditoria_etiquetas_contradictorias.csv", index=False
)

print("\nPRE-FLIGHT DE DATOS: OK")

C:\Users\I21309\AppData\Local\Temp\ipykernel_13552\2832714566.py:138: UserWarning: ES: 1 grupo(s) con etiquetas contradictorias. Se conservan las etiquetas originales y se mantienen juntos todos sus registros. Revisar manualmente antes de interpretar resultados.
  GRUPOS_TRAIN[variante] = construir_grupos(variante, dfs, y_ref)


AUDITORÍA PREVIA — TRAIN Y GROUPED CV


,variante,n_train,n_grupos,n_grupos_multiples,n_filas_en_grupos_multiples,tamano_grupo_max,n_grupos_etiquetas_contradictorias,criterio
0,mx,2399,2393,3,9,5,0,union_texto_normal_stem_lemma_y_original_exacto
1,es,2397,2289,45,153,17,1,union_texto_normal_stem_lemma_y_original_exacto
2,cu,2400,2389,11,22,2,0,union_texto_normal_stem_lemma_y_original_exacto



Solapamientos entre folds (todos deben ser 0):


,variante,nivel,n_grupos_compartidos
0,cu,final,0
1,cu,inner,0
2,cu,outer,0
3,es,final,0
4,es,inner,0
5,es,outer,0
6,mx,final,0
7,mx,inner,0
8,mx,outer,0



[PRE-FLIGHT][ML] Train alineado + Grouped CV: OK


## 4. TF-IDF 10k y constructores de los modelos

Mismo TF-IDF (stopwords NLTK, 10 000 términos, n-gramas `(1,1)`/`(1,2)`), mismos nueve rasgos sin escalar y mismos constructores de LR, RF y XGBoost que en el notebook 10k. Solo se copian los constructores necesarios para reajustar los modelos; en este notebook no se ejecuta ninguna búsqueda.

In [3]:
def crear_preprocesador_tfidf(ngram_range=(1, 1)):
    """Exactamente el TF-IDF base original LR/RF y sus 9 rasgos."""
    tfidf = TfidfVectorizer(stop_words=STOP_WORDS, max_features=10000, ngram_range=ngram_range)
    return ColumnTransformer(
        transformers=[
            ("tfidf", tfidf, "MESSAGE_CLEAN"),
            ("linguisticas", "passthrough", FEATURE_COLS),
        ],
        remainder="drop",
    )


# Única representación compartida por todos los modelos.
crear_preprocesador_lr_rf = crear_preprocesador_tfidf
crear_preprocesador_xgb = crear_preprocesador_tfidf

print(
    "[CONFIG][HardVoting] TF-IDF: stopwords NLTK, max_features=10000, n-gramas (1,1)/(1,2) elegidos en el Inner CV."
)

TF-IDF original LR/RF: stopwords ES, max_features=10000, ngramas=(1,1)/(1,2) elegidos en Inner CV.


In [4]:
def crear_pipeline_lr_rf(modelo):
    """Pipeline TF-IDF + 9 rasgos + LR o RF (class_weight='balanced')."""
    if modelo == "LR":
        clf = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE, class_weight="balanced")
    elif modelo == "RF":
        clf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_jobs=1)
    else:
        raise ValueError(f"Modelo LR/RF no reconocido: {modelo}")
    return Pipeline(steps=[("features", crear_preprocesador_lr_rf()), ("clf", clf)])

In [5]:
# Alias compartido: mismo TF-IDF original LR/RF para LR, RF y XGBoost.
crear_preprocesador = crear_preprocesador_tfidf

In [6]:
# ============================================================
# CONSTRUCTORES DE XGBOOST (sin búsqueda)
# ============================================================


def crear_xgboost(params, random_state=RANDOM_STATE):
    """Clasificador XGBoost con los parámetros fijos del experimento."""
    return XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=N_JOBS_XGB,
        **params,
    )


def crear_pipeline(params, random_state=RANDOM_STATE, ngramas="uni"):
    """Pipeline TF-IDF + 9 rasgos + XGBoost con el rango de n-gramas indicado."""
    if ngramas not in NGRAMAS_OPCIONES:
        raise ValueError(f"Opción de n-gramas no reconocida: {ngramas}")
    return Pipeline(
        steps=[
            ("features", crear_preprocesador(ngram_range=NGRAMAS_OPCIONES[ngramas])),
            ("xgb", crear_xgboost(params, random_state=random_state)),
        ]
    )

## 5. Cargar las configuraciones seleccionadas por el Nested CV 10k

Se leen `outer_folds_ml.csv` (configuración elegida por el Inner CV en cada Outer Fold) y `resumen_nested_ml.csv` (medias de los modelos base) del notebook 10k, y se verifica que existan las 45 configuraciones (3 variantes × 3 modelos × 5 folds). Los modelos se reconstruyen con esas configuraciones; XGBoost usa la semilla `RANDOM_STATE + outer_fold`, igual que en el notebook 10k. No se usan los F1 Outer para escoger hiperparámetros.

In [7]:
# ============================================================
# CARGAR SELECCIONES OUTER DEL EXPERIMENTO 10k
# ============================================================
ruta_outer = BASE_OUT_DIR / "outer_folds_ml.csv"
ruta_resumen = BASE_OUT_DIR / "resumen_nested_ml.csv"
ruta_final = BASE_OUT_DIR / "seleccion_final_ml.csv"

for ruta in [ruta_outer, ruta_resumen]:
    if not Path(ruta).is_file():
        raise FileNotFoundError(
            f"No existe {ruta}. Ejecuta primero ML_TFIDF_10k_GroupNestedCV_5x3_CORREGIDO."
        )

df_outer_base = pd.read_csv(ruta_outer)
df_resumen_base = pd.read_csv(ruta_resumen)


def parse_dict_seguro(valor):
    """Convierte a dict los hiperparámetros guardados como texto en el CSV."""
    if isinstance(valor, dict):
        return valor
    if pd.isna(valor):
        return {}
    return ast.literal_eval(str(valor))


df_outer_base["best_params_dict"] = df_outer_base["best_params"].map(parse_dict_seguro)

esperadas = {(v, m, f) for v in VARIANTES for m in MODELOS for f in range(1, N_OUTER + 1)}
presentes = set(
    zip(df_outer_base["variante"], df_outer_base["modelo"], df_outer_base["outer_fold"].astype(int))
)
faltantes = esperadas - presentes
if faltantes:
    raise ValueError(
        f"Faltan configuraciones Outer en outer_folds_ml.csv: {sorted(faltantes)[:10]}"
    )

print("Configuraciones Outer 10k cargadas:", len(df_outer_base))
display(
    df_outer_base[
        ["variante", "modelo", "outer_fold", "best_preprocessing", "best_ngramas", "outer_f1_macro"]
    ]
    .sort_values(["variante", "modelo", "outer_fold"])
    .head(15)
)

Configuraciones Outer 10k cargadas: 45


,variante,modelo,outer_fold,best_preprocessing,best_ngramas,outer_f1_macro
20,cu,LR,1,stem,uni_bi,0.636764
21,cu,LR,2,stem,uni_bi,0.665840
22,cu,LR,3,stem,uni,0.667616
23,cu,LR,4,stem,uni,0.644630
24,cu,LR,5,stem,uni,0.678176
25,cu,RF,1,stem,uni,0.672619
26,cu,RF,2,stem,uni,0.656005
27,cu,RF,3,normal,uni,0.620124
28,cu,RF,4,stem,uni,0.651758
29,cu,RF,5,normal,uni_bi,0.703551


In [8]:
# ============================================================
# RECONSTRUIR LOS 3 MODELOS DE CADA OUTER FOLD
# SIN REPETIR GridSearch/Optuna
# ============================================================
def fila_config(variante, modelo, outer_fold):
    """Configuración elegida por el Inner CV para un modelo, variante y Outer Fold."""
    q = df_outer_base[
        (df_outer_base["variante"] == variante)
        & (df_outer_base["modelo"] == modelo)
        & (df_outer_base["outer_fold"].astype(int) == int(outer_fold))
    ]
    if len(q) != 1:
        raise ValueError(f"Configuración no única: {variante}/{modelo}/fold{outer_fold}")
    return q.iloc[0]


def ajustar_modelo_outer(variante, modelo, outer_fold, idx_train, idx_test, X_por_prep, y):
    """Reajusta un modelo base sobre el Outer Train y predice el Outer Test."""
    fila = fila_config(variante, modelo, outer_fold)
    prep = fila["best_preprocessing"]
    params = dict(fila["best_params_dict"])

    X_train = X_por_prep[prep].iloc[idx_train].reset_index(drop=True)
    X_test = X_por_prep[prep].iloc[idx_test].reset_index(drop=True)
    y_train = y.iloc[idx_train].reset_index(drop=True)
    y_test = y.iloc[idx_test].reset_index(drop=True)

    if modelo in {"LR", "RF"}:
        estimador = crear_pipeline_lr_rf(modelo)
        estimador.set_params(**params)
    elif modelo == "XGBoost":
        ngramas = fila["best_ngramas"]
        estimador = crear_pipeline(
            params, random_state=RANDOM_STATE + int(outer_fold), ngramas=ngramas
        )
    else:
        raise ValueError(modelo)

    estimador.fit(X_train, y_train)
    pred = np.asarray(estimador.predict(X_test), dtype=int)
    proba = np.asarray(estimador.predict_proba(X_test)[:, 1], dtype=float)

    return {
        "modelo": modelo,
        "prep": prep,
        "pred": pred,
        "proba": proba,
        "y_true": y_test.to_numpy(),
    }

## 6. Evaluación Hard Voting sobre los Outer folds

En cada Outer Fold se reajustan los tres modelos sobre el Outer Train y se predice el Outer Test. La predicción final es 1 si al menos dos modelos predicen 1. Se calculan también las probabilidades de cada modelo, pero la regla de voto usa solo las clases predichas. Se reportan F1-Macro por fold, F1 por clase y F1 OOF global; la comparación con los modelos base es descriptiva (medias y desviaciones), sin prueba estadística.

In [9]:
# ============================================================
# HARD VOTING — LR + RF + XGBOOST
# ============================================================
# Regla fijada ANTES de ver el Outer Test:
# gana la clase predicha por al menos 2 de los 3 modelos.
#
# Con 3 modelos no existen empates.
# No se prueban subconjuntos con el Outer Test.

FILAS_HARD = []
OOF_HARD = []

for variante in VARIANTES:
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    grupos = grupos_train(variante)
    outer_splits = crear_splits_outer(y, grupos, variante)

    for outer_fold, (idx_train, idx_test) in enumerate(outer_splits, start=1):
        resultados = {
            modelo: ajustar_modelo_outer(
                variante, modelo, outer_fold, idx_train, idx_test, X_por_prep, y
            )
            for modelo in MODELOS
        }

        y_true = resultados["LR"]["y_true"]
        for m in MODELOS[1:]:
            assert np.array_equal(y_true, resultados[m]["y_true"])

        p_lr = resultados["LR"]["pred"]
        p_rf = resultados["RF"]["pred"]
        p_xgb = resultados["XGBoost"]["pred"]

        pred_hard = ((p_lr + p_rf + p_xgb) >= 2).astype(int)

        f1_c0, f1_c1 = f1_score(y_true, pred_hard, labels=[0, 1], average=None, zero_division=0)
        f1_macro = f1_score(y_true, pred_hard, average="macro")

        FILAS_HARD.append(
            {
                "variante": variante,
                "modelo": "HardVoting",
                "outer_fold": outer_fold,
                "outer_f1_clase_0": float(f1_c0),
                "outer_f1_clase_1": float(f1_c1),
                "outer_f1_macro": float(f1_macro),
                "regla": "mayoria_2_de_3",
            }
        )

        OOF_HARD.append(
            pd.DataFrame(
                {
                    "variante": variante,
                    "outer_fold": outer_fold,
                    "idx_train_oficial": idx_test,
                    "y_true": y_true,
                    "pred_lr": p_lr,
                    "pred_rf": p_rf,
                    "pred_xgb": p_xgb,
                    "y_pred": pred_hard,
                }
            )
        )

        print(
            f"[{variante.upper()}][HardVoting][Outer {outer_fold}/{N_OUTER}] "
            f"F1-Macro={f1_macro:.4f}"
        )

df_hard_outer = pd.DataFrame(FILAS_HARD)
df_hard_oof = pd.concat(OOF_HARD, ignore_index=True)

df_hard_resumen = df_hard_outer.groupby(["variante", "modelo"], as_index=False).agg(
    f1_nested_mean=("outer_f1_macro", "mean"),
    f1_nested_std=("outer_f1_macro", lambda s: s.std(ddof=1)),
)

oof_rows = []
for variante, g in df_hard_oof.groupby("variante"):
    g = g.sort_values("idx_train_oficial")
    f1_c0, f1_c1 = f1_score(g.y_true, g.y_pred, labels=[0, 1], average=None, zero_division=0)
    oof_rows.append(
        {
            "variante": variante,
            "modelo": "HardVoting",
            "f1_oof_global": f1_score(g.y_true, g.y_pred, average="macro"),
            "f1_oof_clase_0": f1_c0,
            "f1_oof_clase_1": f1_c1,
        }
    )
df_hard_oof_resumen = pd.DataFrame(oof_rows)

print("\n[HardVoting] RESUMEN NESTED CV")
display(df_hard_resumen.round(4))

print("\nHARD VOTING — DETALLE POR FOLD")
display(df_hard_outer.round(4))

print("\nHARD VOTING — OOF GLOBAL")
display(df_hard_oof_resumen.round(4))

comparacion = pd.concat(
    [
        df_resumen_base[["variante", "modelo", "f1_nested_mean", "f1_nested_std"]],
        df_hard_resumen[["variante", "modelo", "f1_nested_mean", "f1_nested_std"]],
    ],
    ignore_index=True,
).sort_values(["variante", "modelo"])

print("\nCOMPARACIÓN DESCRIPTIVA — MODELOS BASE + HARD VOTING")
display(comparacion.round(4))

df_hard_outer.to_csv(VOTING_OUT_DIR / "hard_voting_outer_folds.csv", index=False)
df_hard_oof.to_csv(VOTING_OUT_DIR / "hard_voting_oof.csv", index=False)
df_hard_resumen.to_csv(VOTING_OUT_DIR / "hard_voting_resumen.csv", index=False)
df_hard_oof_resumen.to_csv(VOTING_OUT_DIR / "hard_voting_oof_resumen.csv", index=False)

[MX][HardVoting][Outer 1/5] F1-Macro=0.6397
[MX][HardVoting][Outer 2/5] F1-Macro=0.6109
[MX][HardVoting][Outer 3/5] F1-Macro=0.6338
[MX][HardVoting][Outer 4/5] F1-Macro=0.6399
[MX][HardVoting][Outer 5/5] F1-Macro=0.6409
[ES][HardVoting][Outer 1/5] F1-Macro=0.7158
[ES][HardVoting][Outer 2/5] F1-Macro=0.7039
[ES][HardVoting][Outer 3/5] F1-Macro=0.7311
[ES][HardVoting][Outer 4/5] F1-Macro=0.7086
[ES][HardVoting][Outer 5/5] F1-Macro=0.6819
[CU][HardVoting][Outer 1/5] F1-Macro=0.6648
[CU][HardVoting][Outer 2/5] F1-Macro=0.6478
[CU][HardVoting][Outer 3/5] F1-Macro=0.6393
[CU][HardVoting][Outer 4/5] F1-Macro=0.6564
[CU][HardVoting][Outer 5/5] F1-Macro=0.6986

HARD VOTING — NESTED OUTER


,variante,modelo,f1_nested_mean,f1_nested_std
0,cu,HardVoting,0.6614,0.0229
1,es,HardVoting,0.7082,0.0180
2,mx,HardVoting,0.6330,0.0127



HARD VOTING — DETALLE POR FOLD


,variante,modelo,outer_fold,outer_f1_clase_0,outer_f1_clase_1,outer_f1_macro,regla
0,mx,HardVoting,1,0.7621,0.5174,0.6397,mayoria_2_de_3
1,mx,HardVoting,2,0.7406,0.4812,0.6109,mayoria_2_de_3
2,mx,HardVoting,3,0.7597,0.5079,0.6338,mayoria_2_de_3
3,mx,HardVoting,4,0.7764,0.5034,0.6399,mayoria_2_de_3
4,mx,HardVoting,5,0.7571,0.5247,0.6409,mayoria_2_de_3
5,es,HardVoting,1,0.7894,0.6423,0.7158,mayoria_2_de_3
6,es,HardVoting,2,0.8104,0.5974,0.7039,mayoria_2_de_3
7,es,HardVoting,3,0.8338,0.6284,0.7311,mayoria_2_de_3
8,es,HardVoting,4,0.7889,0.6282,0.7086,mayoria_2_de_3
9,es,HardVoting,5,0.7762,0.5875,0.6819,mayoria_2_de_3



HARD VOTING — OOF GLOBAL


,variante,modelo,f1_oof_global,f1_oof_clase_0,f1_oof_clase_1
0,cu,HardVoting,0.6617,0.7970,0.5264
1,es,HardVoting,0.7088,0.8004,0.6172
2,mx,HardVoting,0.6331,0.7593,0.5070



COMPARACIÓN DESCRIPTIVA — MODELOS BASE + HARD VOTING


,variante,modelo,f1_nested_mean,f1_nested_std
9,cu,HardVoting,0.6614,0.0229
0,cu,LR,0.6586,0.0172
1,cu,RF,0.6608,0.0305
2,cu,XGBoost,0.6437,0.0170
10,es,HardVoting,0.7082,0.0180
3,es,LR,0.7024,0.0157
4,es,RF,0.6959,0.0298
5,es,XGBoost,0.6847,0.0163
11,mx,HardVoting,0.6330,0.0127
6,mx,LR,0.6450,0.0148


### 6.1 Figura: comparación de los modelos de ML en la Nested CV

Resume el resultado principal de ML **solo con train** (Nested CV 5×3): F1-Macro y F1 de la clase irónica de LR, RF, XGBoost y Hard Voting, con un grupo por variante.

- punto grande: media de los 5 Outer Folds;
- línea vertical: ± desviación estándar entre folds;
- puntos pequeños: cada Outer Fold.

Se usan puntos en lugar de barras porque los F1 están entre 0.45 y 0.75: unas barras que empiecen en 0 ocultarían las diferencias y unas que empiecen más arriba las exagerarían. Solo lee CSV ya guardados; no entrena nada.

In [ ]:
# ============================================================
# FIGURA — F1 DE LA NESTED CV DE LOS MODELOS DE ML (solo train)
# ============================================================
import matplotlib.pyplot as plt

cols_fig = ["variante", "modelo", "outer_fold", "outer_f1_macro", "outer_f1_clase_1"]
folds_base = pd.read_csv(BASE_OUT_DIR / "outer_folds_ml.csv")
folds_hv = pd.read_csv(VOTING_OUT_DIR / "hard_voting_outer_folds.csv")
folds_ml = pd.concat([folds_base[cols_fig], folds_hv[cols_fig]], ignore_index=True)

ORDEN_ML = ["LR", "RF", "XGBoost", "HardVoting"]
ETIQUETA_ML = {"LR": "LR", "RF": "RF", "XGBoost": "XGBoost", "HardVoting": "Hard Voting"}
COLOR_ML = {"LR": "#4c72b0", "RF": "#55a868", "XGBoost": "#c44e52", "HardVoting": "#8172b3"}
METRICAS_FIG = [("outer_f1_macro", "F1-Macro"), ("outer_f1_clase_1", "F1 de la clase irónica")]

rng_fig = np.random.default_rng(RANDOM_STATE)  # solo para separar los puntos de cada fold
separacion = 0.18

fig, axes = plt.subplots(1, len(METRICAS_FIG), figsize=(13, 4.8), squeeze=False)
filas_resumen = []
for ax, (col, titulo) in zip(axes[0], METRICAS_FIG):
    for k, modelo in enumerate(ORDEN_ML):
        for j, variante in enumerate(VARIANTES):
            valores = folds_ml.loc[
                (folds_ml["variante"] == variante) & (folds_ml["modelo"] == modelo), col
            ].to_numpy()
            if len(valores) == 0:
                continue
            x = j + (k - (len(ORDEN_ML) - 1) / 2) * separacion
            media, std = valores.mean(), valores.std(ddof=1)
            ax.scatter(
                x + rng_fig.uniform(-0.03, 0.03, len(valores)),
                valores,
                s=14,
                facecolors="none",
                edgecolors=COLOR_ML[modelo],
                alpha=0.8,
                zorder=2,
            )
            ax.errorbar(
                x,
                media,
                yerr=std,
                fmt="o",
                ms=8,
                capsize=4,
                lw=1.5,
                color=COLOR_ML[modelo],
                zorder=3,
                label=ETIQUETA_ML[modelo] if j == 0 else None,
            )
            filas_resumen.append(
                {
                    "metrica": col,
                    "variante": variante,
                    "modelo": modelo,
                    "media": media,
                    "desviacion": std,
                    "n_folds": len(valores),
                }
            )
    ax.set_xticks(range(len(VARIANTES)), [v.upper() for v in VARIANTES])
    ax.set_title(titulo)
    ax.set_ylabel(titulo)
    ax.grid(axis="y", alpha=0.25)
axes[0, 0].legend(fontsize=9, loc="best")
fig.suptitle(
    "Modelos de ML — Nested CV 5×3 (media ± d.e. de 5 Outer Folds; puntos huecos = cada fold)",
    fontsize=12,
)
fig.tight_layout()
fig.savefig(VOTING_OUT_DIR / "ml_nested_cv_f1.png", dpi=200, bbox_inches="tight")
fig.savefig(VOTING_OUT_DIR / "ml_nested_cv_f1.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)

df_resumen_fig_ml = pd.DataFrame(filas_resumen)
df_resumen_fig_ml.to_csv(VOTING_OUT_DIR / "ml_nested_cv_f1_figura.csv", index=False)
display(
    df_resumen_fig_ml.pivot_table(
        index=["variante", "modelo"], columns="metrica", values=["media", "desviacion"]
    ).round(4)
)

## 7. Test oficial — bloqueado

Permanece bloqueado por defecto: requiere `MODO="completo"`, `EJECUTAR_TEST_OFICIAL=True` y `CONFIRMACION_TEST="TEST_OFICIAL_CONGELADO"`. Lee `seleccion_final_ml.csv` y reutiliza los nueve modelos finales ya entrenados por el notebook 10k (`modelos_finales/{lr|rf|xgboost}_{variante}_{preprocessing}.pkl`); no entrena ni optimiza nada. Cada modelo carga el test en su propio preprocesamiento y se verifica que IDs y etiquetas estén alineados entre los tres antes de votar. Guarda las métricas en `hard_voting_test_oficial.csv` y las predicciones por texto en `hard_voting_test_predicciones.csv` (para las matrices de confusión de `05_comparacion_final_test.ipynb`; Hard Voting no produce probabilidades, por lo que no entra en la ROC).

In [10]:
# ============================================================
# MODELOS FINALES YA ENTRENADOS POR EL NOTEBOOK 10k
# ============================================================
if Path(ruta_final).is_file():
    df_final = pd.read_csv(ruta_final)
else:
    df_final = None


def cargar_modelo_final(variante, modelo):
    """Carga un modelo final entrenado por el notebook ML."""
    if df_final is None:
        raise FileNotFoundError(
            "No existe seleccion_final_ml.csv. Ejecuta primero la fase final del notebook 10k."
        )
    q = df_final[(df_final["variante"] == variante) & (df_final["modelo"] == modelo)]
    if len(q) != 1:
        raise ValueError(f"Selección final no única: {variante}/{modelo}")
    prep = q.iloc[0]["preprocessing"]

    if modelo == "LR":
        nombre = f"lr_{variante}_{prep}.pkl"
    elif modelo == "RF":
        nombre = f"rf_{variante}_{prep}.pkl"
    elif modelo == "XGBoost":
        nombre = f"xgboost_{variante}_{prep}.pkl"
    else:
        raise ValueError(modelo)

    ruta = BASE_MODELS_DIR / nombre
    if not Path(ruta).is_file():
        raise FileNotFoundError(f"No existe el modelo final: {ruta}")

    return joblib.load(ruta), prep


def obtener_prediccion_test(variante, modelo):
    """IDs, etiquetas, predicciones y probabilidades del test oficial para un modelo final."""
    estimador, prep = cargar_modelo_final(variante, modelo)
    df_test = cargar_split(variante, prep, split="test")
    X_test, y_test = preparar_xy(df_test)
    ids = (
        df_test["ID"].astype(str).reset_index(drop=True)
        if "ID" in df_test.columns
        else pd.Series(np.arange(len(df_test)).astype(str))
    )
    pred = np.asarray(estimador.predict(X_test), dtype=int)
    proba = np.asarray(estimador.predict_proba(X_test)[:, 1], dtype=float)
    return ids, y_test.to_numpy(), pred, proba

### 7.1 Configuraciones finales usadas por el Hard Voting

Muestra el preprocesamiento y los n-gramas de los tres modelos finales (entrenados sobre todo el train) por variante. Usa `df_final`, definido en la celda anterior, y no consulta el test.

In [12]:
# ============================================================
# RESUMEN DE CONFIGURACIONES FINALES DEL HARD VOTING
# ============================================================

columnas_config = ["variante", "modelo", "preprocessing"]

# Si existe la columna de n-gramas, la incluimos.
if "ngramas" in df_final.columns:
    columnas_config.append("ngramas")

df_config_hard = (
    df_final[columnas_config].sort_values(["variante", "modelo"]).reset_index(drop=True)
)

print("CONFIGURACIONES FINALES UTILIZADAS POR HARD VOTING")
display(df_config_hard)

print("\nRESUMEN POR VARIANTE")

for variante in VARIANTES:
    print(f"\n{variante.upper()}")

    df_v = df_config_hard[df_config_hard["variante"] == variante]

    for _, fila in df_v.iterrows():
        texto = f"  {fila['modelo']}: preprocessing={fila['preprocessing']}"

        if "ngramas" in df_config_hard.columns:
            texto += f" | ngramas={fila['ngramas']}"

        print(texto)

    print("  → Hard Voting: mayoría de 2 de 3")

CONFIGURACIONES FINALES UTILIZADAS POR HARD VOTING


,variante,modelo,preprocessing,ngramas
0,cu,LR,stem,uni_bi
1,cu,RF,stem,uni
2,cu,XGBoost,stem,uni_bi
3,es,LR,lemma,uni_bi
4,es,RF,stem,uni
5,es,XGBoost,stem,uni
6,mx,LR,normal,uni_bi
7,mx,RF,lemma,uni
8,mx,XGBoost,lemma,uni



RESUMEN POR VARIANTE

MX
  LR: preprocessing=normal | ngramas=uni_bi
  RF: preprocessing=lemma | ngramas=uni
  XGBoost: preprocessing=lemma | ngramas=uni
  → Hard Voting: mayoría de 2 de 3

ES
  LR: preprocessing=lemma | ngramas=uni_bi
  RF: preprocessing=stem | ngramas=uni
  XGBoost: preprocessing=stem | ngramas=uni
  → Hard Voting: mayoría de 2 de 3

CU
  LR: preprocessing=stem | ngramas=uni_bi
  RF: preprocessing=stem | ngramas=uni
  XGBoost: preprocessing=stem | ngramas=uni_bi
  → Hard Voting: mayoría de 2 de 3


In [11]:
# ============================================================
# TEST OFICIAL — HARD VOTING (BLOQUEADO POR DEFECTO)
# ============================================================
if not EJECUTAR_TEST_OFICIAL:
    print("TEST OFICIAL BLOQUEADO. No se ha consultado para decidir el ensemble.")
else:
    if MODO != "completo":
        raise ValueError("El test oficial requiere MODO='completo'.")
    if CONFIRMACION_TEST != "TEST_OFICIAL_CONGELADO":
        raise ValueError(
            "Bloqueado: confirma explícitamente la evaluación final con CONFIRMACION_TEST."
        )
    filas_test = []
    predicciones_hv = []
    for variante in VARIANTES:
        datos = {modelo: obtener_prediccion_test(variante, modelo) for modelo in MODELOS}

        ids_ref, y_ref, _, _ = datos["LR"]
        for m in MODELOS[1:]:
            ids_m, y_m, _, _ = datos[m]
            assert ids_ref.equals(ids_m), f"IDs test desalineados: {variante}/{m}"
            assert np.array_equal(y_ref, y_m), f"Etiquetas test desalineadas: {variante}/{m}"

        pred = ((datos["LR"][2] + datos["RF"][2] + datos["XGBoost"][2]) >= 2).astype(int)

        f1_c0, f1_c1 = f1_score(y_ref, pred, labels=[0, 1], average=None, zero_division=0)
        predicciones_hv.append(
            pd.DataFrame(
                {
                    "variante": variante,
                    "modelo": "HardVoting",
                    "ID": ids_ref.astype(str).to_numpy(),
                    "y_true": y_ref,
                    "y_pred": pred,
                }
            )
        )
        filas_test.append(
            {
                "variante": variante,
                "modelo": "HardVoting",
                "f1_macro_test": f1_score(y_ref, pred, average="macro"),
                "accuracy_test": accuracy_score(y_ref, pred),
                "precision_macro_test": precision_score(
                    y_ref, pred, average="macro", zero_division=0
                ),
                "recall_macro_test": recall_score(y_ref, pred, average="macro", zero_division=0),
                "f1_clase_0": f1_c0,
                "f1_clase_1": f1_c1,
            }
        )

    df_hard_test = pd.DataFrame(filas_test)
    display(df_hard_test.round(4))
    df_hard_test.to_csv(VOTING_OUT_DIR / "hard_voting_test_oficial.csv", index=False)
    # Predicciones por texto para las matrices de confusión (Hard Voting no produce probabilidades).
    pd.concat(predicciones_hv, ignore_index=True).to_csv(
        VOTING_OUT_DIR / "hard_voting_test_predicciones.csv", index=False
    )
    print("TEST OFICIAL EVALUADO. No usar estos resultados para cambiar la combinación de modelos.")

TEST OFICIAL BLOQUEADO. No se ha consultado para decidir el ensemble.


## 8. Resumen metodológico

- El ensemble contiene siempre **LR + RF + XGBoost**, con regla de mayoría 2 de 3.
- No se seleccionan pesos ni subconjuntos usando los resultados Outer.
- El F1-Macro medio de los cinco Outer folds es la referencia de desarrollo; la comparación con los modelos base es descriptiva.
- El test oficial permanece reservado hasta cerrar definitivamente el procedimiento.
- Si el Voting no supera a los modelos individuales, se reporta tal cual; no se reajusta mirando el Outer/Test.